### Open ended stub antenna

An open-ended stub antenna is a simple radiating element consisting of a transmission line (typically microstrip or coaxial cable) with one end open-circuited, allowing electromagnetic energy to radiate directly into free space.

In [1]:
import gmsh
import math
import os
import json

from palacetoolkit.viz import view_mesh
from palacetoolkit.geometry import extract_tag, xmin, xmax, ymin, ymax, zmin, zmax
from palacetoolkit.simulation import Simulation

### Paramters
- l1 : Ground plane length along x-axis, specified as a scalar in meters
- w1 : Ground plane width along y-axis, specified as a scalar in meters
- h : Patch height along z-axis, specified as a scalar in meters.

- strip_line_length : Notch length along x-axis, specified as a scalar in meters. 
- strip_lined_width: Notch width along y-axis, specified as a scalar in meters.
- air_height : Air box height along z-axis, specified as a scalar in meters.  
- air_margin : Air box margin along x and y axes, specified as a scalar in meters.
- freq  : Simulation frequency in GHz, specified as a scalar.
- filename : Output mesh filename, specified as a string.

In [2]:
l1: float = 0.06
w1: float = 0.06
strip_line_length: float = 0.04
strip_line_width: float = 0.002
h: float = 0.0013
air_height: float = 0.025    
air_margin: float = 0.025    
freq: float = 3.3
filename: str = "sw_antenna.msh"

### Model initialization

In [3]:
gmsh.initialize()
gmsh.model.add("patch_antenna")
kernel = gmsh.model.occ

In [4]:
# Total domain bounds
total_xmin = -l1/2 - air_margin
total_xmax = l1/2 + air_margin
total_ymin = -w1/2 - air_margin
total_ymax = w1/2 + air_margin
total_zmax = h + air_height

substrate = kernel.addBox(-l1/2, -w1/2, 0, l1, w1, h)

ground_plane = kernel.addRectangle(-l1/2, -w1/2, 0, l1, w1)

strip_line_1 = kernel.addRectangle(-l1/2, -strip_line_width/2, h, strip_line_length, strip_line_width)

gap = 0
lumped_port = kernel.addRectangle(-l1/2 + gap, -strip_line_width/2, 0, h - gap, strip_line_width)
kernel.rotate([(2, lumped_port)], -l1/2, 0, 0, 0, 1, 0, -math.pi/2)
kernel.synchronize()

air_box = kernel.addBox(
    total_xmin, total_ymin, 0,
    total_xmax - total_xmin,
    total_ymax - total_ymin,
    total_zmax
)
kernel.synchronize()

all_tools = [(2, ground_plane), (2, strip_line_1), (2, lumped_port)]

result, result_map = kernel.fragment(
    [(3, substrate), (3, air_box)], 
    all_tools
)
kernel.synchronize()
kernel.removeAllDuplicates()
kernel.synchronize()

### Geometry

In [5]:
# regions
all_2d = gmsh.model.getEntities(2)
all_3d = gmsh.model.getEntities(3)

tol = 1e-6

ground_tags = []
sides = []
patch_tags = []
port_tags = []
farfield_tags = []
substrate_tags = []
air_tags = []

# 2D surfaces
for dim, tag in all_2d:
    bbox = gmsh.model.getBoundingBox(dim, tag)
    xmin, ymin, zmin = bbox[0], bbox[1], bbox[2]
    xmax, ymax, zmax = bbox[3], bbox[4], bbox[5]
    
    # Ground plane
    if (math.isclose(zmin, 0, abs_tol=tol) and 
        math.isclose(zmax, 0, abs_tol=tol) and
        xmin >= -l1/2 - tol and xmax <= l1/2 + tol):
        ground_tags.append(tag)
        
    # Patch
    elif (math.isclose(zmin, h, abs_tol=tol) and 
            math.isclose(zmax, h, abs_tol=tol) and
            math.isclose(ymax, strip_line_width/2, abs_tol=tol)):
        patch_tags.append(tag)
        
    # Lumped port
    elif (math.isclose(xmin, -l1/2, abs_tol=tol) and 
            math.isclose(xmax, -l1/2, abs_tol=tol) and
            math.isclose(ymin, -strip_line_width/2, abs_tol=tol) and
            math.isclose(ymax, strip_line_width/2, abs_tol=tol) and
            zmin >= gap - tol and zmax <= h + tol):
        port_tags.append(tag)
        
    # Far-field (outer air box surfaces)
    elif (math.isclose(xmin, -l1/2 - air_margin, abs_tol=tol) or
            math.isclose(xmax, l1/2 + air_margin, abs_tol=tol) or
            math.isclose(ymin, -w1/2 - air_margin, abs_tol=tol) or
            math.isclose(ymax, w1/2 + air_margin, abs_tol=tol) or
            math.isclose(zmax, h + air_height, abs_tol=tol)):
        farfield_tags.append(tag)
    else:
        sides.append(tag)

# 3D volumes
for dim, tag in all_3d:
    bbox = gmsh.model.getBoundingBox(dim, tag)
    zmax = bbox[5]
    
    if math.isclose(zmax, h, abs_tol=tol):
        substrate_tags.append(tag)
    else:
        air_tags.append(tag)

# physical groups
pg_map = {
}

if substrate_tags:
    pg_map["substrate"] = gmsh.model.addPhysicalGroup(3, substrate_tags, name = "Substrate")
if air_tags:
    pg_map["air"] = gmsh.model.addPhysicalGroup(3, air_tags, name = "Air")
if ground_tags:
    pg_map["ground_plane"] = gmsh.model.addPhysicalGroup(2, ground_tags, name = "GroundPlane")
if patch_tags:
    pg_map["patch"] = gmsh.model.addPhysicalGroup(2, patch_tags, name = "Patch")
if port_tags:
    pg_map["lumped_port"] = gmsh.model.addPhysicalGroup(2, port_tags, name = "LumpedPort")
if farfield_tags:
    pg_map["farfield"] = gmsh.model.addPhysicalGroup(2, farfield_tags, name = "FarField")
if sides:
    pg_map["sides"] = gmsh.model.addPhysicalGroup(2, sides, name = "Sides")

### Mesh generation and basic refinement

In [6]:
wavelength = 3e8 / (freq * 1e9)

sim = Simulation()
sim.set_output_dir("./palace-sim-example")

refine_dimtags = [(2, t) for t in (port_tags + patch_tags)]
sim.refine_near_surfaces(
    refine_dimtags,
    wavelength,
    ppw_near=20,
    ppw_far=8,
    transition_distance=wavelength / 6,
    set_as_background=True,
 )

gmsh.model.mesh.generate(3)
gmsh.model.mesh.setOrder(1)

mesh_path = os.path.join(str(sim.output_dir), filename)
gmsh.write(mesh_path)

gmsh.fltk.run()
gmsh.finalize()

  7 conductor boundary curves
  ppw_near=20  ppw_far=8
  SizeMin=0.0045 (20 pts/λ)
  SizeMax=0.0114 (8 pts/λ)
  Transition distance: 0 → 0.0152
Info    : Meshing 1D...
Info    : [  0%] Meshing curve 24 (Line)
Info    : [ 10%] Meshing curve 25 (Line)
Info    : [ 10%] Meshing curve 26 (Line)
Info    : [ 10%] Meshing curve 27 (Line)
Info    : [ 20%] Meshing curve 28 (Line)
Info    : [ 20%] Meshing curve 29 (Line)
Info    : [ 20%] Meshing curve 30 (Line)
Info    : [ 30%] Meshing curve 31 (Line)
Info    : [ 30%] Meshing curve 32 (Line)
Info    : [ 30%] Meshing curve 33 (Line)
Info    : [ 40%] Meshing curve 34 (Line)
Info    : [ 40%] Meshing curve 35 (Line)
Info    : [ 40%] Meshing curve 36 (Line)
Info    : [ 40%] Meshing curve 37 (Line)
Info    : [ 50%] Meshing curve 38 (Line)
Info    : [ 50%] Meshing curve 39 (Line)
Info    : [ 50%] Meshing curve 40 (Line)
Info    : [ 60%] Meshing curve 41 (Line)
Info    : [ 60%] Meshing curve 42 (Line)
Info    : [ 60%] Meshing curve 43 (Line)
Info    : [ 

Info    : Meshing 3D...
Info    : 3D Meshing 2 volumes with 1 connected component
Info    : Tetrahedrizing 452 nodes...
Info    : Done tetrahedrizing 460 nodes (Wall 0.00300749s, CPU 0.002258s)
Info    : Reconstructing mesh...
Info    :  - Creating surface mesh
Info    :  - Identifying boundary edges
Info    :  - Recovering boundary
Info    : Done reconstructing mesh (Wall 0.00608383s, CPU 0.004847s)
Info    : Found volume 2
Info    : Found volume 1
Info    : It. 0 - 0 nodes created - worst tet radius 2.06731 (nodes removed 0 0)
Info    : 3D refinement terminated (508 nodes total):
Info    :  - 0 Delaunay cavities modified for star shapeness
Info    :  - 24 nodes could not be inserted
Info    :  - 1869 tetrahedra created in 0.00186403 sec. (1002664 tets/s)
Info    : 0 node relocations
Info    : Done meshing 3D (Wall 0.0121797s, CPU 0.011042s)
Info    : Optimizing mesh...
Info    : Optimizing volume 1
Info    : Optimization starts (volume = 4.68e-06) with worst = 0.0435933 / average = 0

### Generate palace JSON config

In [7]:
config_filename: str = "palace.conf"
freq_min: float = 3.0
freq_max: float = 3.5
freq_step: float = 0.005
eps_r: float = 2.2
loss_tan: float = 0.0009
port_impedance: float = 50.0
solver_order: int = 2

In [8]:
def attr(name):
    return [pg_map[name]] if name in pg_map else []

sim.set_mesh_file(f"/work/{filename}")
sim.set_config_option("Problem.Output", "/work/results/open_ended_antenna/")

sim.set_config_option("Domains.Materials", [
    {
        "Attributes": attr("substrate"),
        "Permittivity": eps_r,
        "Permeability": 1.0,
        "LossTan": loss_tan
    },
    {
        "Attributes": attr("air"),
        "Permittivity": 1.0,
        "Permeability": 1.0
    }
])

sim.set_config_option("Boundaries.PEC", {
    "Attributes": attr("ground_plane") + attr("patch")
})
sim.set_config_option("Boundaries.LumpedPort", [
    {
        "Index": 1,
        "Attributes": attr("lumped_port"),
        "R": port_impedance,
        "Excitation": True,
        "Direction": "+Z"
    }
])
sim.set_config_option("Boundaries.Absorbing", {
    "Attributes": attr("farfield"),
    "Order": 1
})

sim.set_config_option("Solver.Order", solver_order)
sim.set_config_option("Solver.Driven.MinFreq", freq_min)
sim.set_config_option("Solver.Driven.MaxFreq", freq_max)
sim.set_config_option("Solver.Driven.FreqStep", freq_step)
sim.set_config_option("Solver.Driven.AdaptiveTol", 0.001)

config_path = sim.write_config(config_filename)
print(f"Palace config written to {config_path}")

Palace config written to palace-sim-example/palace.conf
Palace config written to palace-sim-example/palace.conf
